# NB_hospital_daily_summary
**Silver → Gold | CareSync Health Network** (per STTM v1.0)

* **Target:** `adb_caresync.gold.hospital_daily_summary`
* **Grain:** one row per hospital per calendar day
* **Driving table:** `adb_caresync.silver.lab_results` grouped by hospital_id, test_date
* **Joins:** LEFT JOIN `hospitals` on hospital_id; LEFT JOIN `calendar` on date_key = date_format(test_date, 'yyyyMMdd')
* **Window:** LAG(total_tests) partitioned by hospital_id, ordered by summary_date (day-over-day trend)
* **Load:** FULL — CREATE OR REPLACE TABLE ... AS SELECT

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab-test volumes, rates and day-over-day trend (STTM v1.0)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                              AS hospital_id,
  h.hospital_name,
  w.summary_date,
  cal.day_name,
  cal.is_weekend,
  CAST(w.total_tests AS INT)                                              AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                   AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                      AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
  CAST(w.abnormal_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))     AS abnormal_rate_pct,
  CAST(w.critical_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))     AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                     AS prev_day_total_tests,
  CAST((w.total_tests - w.prev_day_total_tests) * 100.0
       / NULLIF(w.prev_day_total_tests, 0) AS DECIMAL(6,2))               AS test_volume_change_pct
FROM with_lag w
LEFT JOIN adb_caresync.silver.hospitals h ON w.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar cal
       ON cal.date_key = CAST(DATE_FORMAT(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM adb_caresync.gold.hospital_daily_summary)                                   AS gold_rows,
  (SELECT COUNT(DISTINCT hospital_id, summary_date) FROM adb_caresync.gold.hospital_daily_summary)  AS gold_distinct_grain,
  (SELECT SUM(total_tests) FROM adb_caresync.gold.hospital_daily_summary)                           AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results)                                            AS silver_lab_results,
  (SELECT COUNT(*) FROM adb_caresync.gold.hospital_daily_summary WHERE day_name IS NULL)            AS rows_missing_calendar;

In [0]:
record_count = spark.sql("SELECT COUNT(*) AS record_count FROM adb_caresync.gold.hospital_daily_summary").first()["record_count"]

dbutils.notebook.exit(record_count)